# 02 - Generate COTs

Generate chain-of-thought reasoning for all GSM8K problems using Qwen3-4B.
Also run the **no_cot** baseline (direct answer, no reasoning).

Outputs:
- `cache/cots/{problem_id}.json` - one file per problem with COT
- `cache/prefills/no_cot_{problem_id}.json` - no_cot baseline results

In [5]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/11-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "https://github.com/JackHopkins/legibility.git", str(REPO_DIR)], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, PARAPHRASE_CACHE, PREFILL_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

Already up to date.


In [6]:
import json
import re
from tqdm import tqdm
from vllm import LLM, SamplingParams
from lib.data import load_gsm8k, extract_predicted_answer
from lib.prompts import build_cot_messages, build_no_cot_messages

## Load Dataset

In [7]:
dataset = load_gsm8k()
print(f"Loaded {len(dataset)} GSM8K problems")
print(f"Example: {dataset[0]['question'][:100]}...")
print(f"Gold answer: {dataset[0]['gold_answer']}")

Loaded 1319 GSM8K problems
Example: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for ...
Gold answer: 18


## Load Model

In [ ]:
llm = LLM(
    model=MODEL_NAME,
    dtype="bfloat16",
    max_model_len=4096,
)
print(f"Loaded {MODEL_NAME}")

## Generate COTs

Use Qwen3 thinking mode to generate step-by-step reasoning.
Cache one file per problem for resumability.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Resume: check which problems are already done
done_ids = {int(p.stem) for p in COT_CACHE.glob("*.json")}
todo = [ex for ex in dataset if ex["problem_id"] not in done_ids]
print(f"Resuming: {len(done_ids)} done, {len(todo)} remaining")

In [ ]:
CHUNK_SIZE = 64
sampling_cot = SamplingParams(temperature=TEMPERATURE, max_tokens=MAX_COT_TOKENS)

for i in tqdm(range(0, len(todo), CHUNK_SIZE), desc="Generating COTs"):
    chunk = todo[i : i + CHUNK_SIZE]

    # Build prompts using chat template
    prompts = []
    for ex in chunk:
        messages = build_cot_messages(ex["question"])
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=True,
        )
        prompts.append(prompt)

    outputs = llm.generate(prompts, sampling_cot)

    for ex, output in zip(chunk, outputs):
        full_response = output.outputs[0].text

        # Extract thinking content
        think_match = re.search(r"<think>(.*?)</think>", full_response, re.DOTALL)
        if think_match:
            cot_text = think_match.group(1).strip()
            answer_portion = full_response[think_match.end():].strip()
        else:
            # Fallback: everything before "The answer is" is COT
            parts = re.split(r"The answer is:?", full_response, maxsplit=1)
            cot_text = parts[0].strip()
            answer_portion = parts[1].strip() if len(parts) > 1 else ""

        predicted = extract_predicted_answer(full_response)

        result = {
            "problem_id": ex["problem_id"],
            "question": ex["question"],
            "gold_answer": ex["gold_answer"],
            "cot_text": cot_text,
            "full_response": full_response,
            "answer_portion": answer_portion,
            "predicted_answer": predicted,
            "error": None,
        }
        (COT_CACHE / f"{ex['problem_id']}.json").write_text(json.dumps(result))

print("COT generation complete.")

## Verify COT Quality

In [ ]:
# Load all COTs and compute accuracy
all_cots = []
for p in sorted(COT_CACHE.glob("*.json"), key=lambda x: int(x.stem)):
    all_cots.append(json.loads(p.read_text()))

correct = sum(1 for c in all_cots if c["predicted_answer"] == c["gold_answer"])
total = len(all_cots)
print(f"COT accuracy (normal condition): {correct}/{total} = {correct/total:.1%}")

# Show a few examples
for c in all_cots[:3]:
    print(f"\n--- Problem {c['problem_id']} ---")
    print(f"Question: {c['question'][:100]}...")
    print(f"Gold: {c['gold_answer']}, Predicted: {c['predicted_answer']}")
    print(f"COT length: {len(c['cot_text'])} chars")

## Generate No-COT Baseline

Same model, same questions, but instructed to answer directly without reasoning.

In [ ]:
# Resume check for no_cot
done_no_cot = {int(p.stem.split("_")[-1]) for p in PREFILL_CACHE.glob("no_cot_*.json")}
todo_no_cot = [ex for ex in dataset if ex["problem_id"] not in done_no_cot]
print(f"No-COT: {len(done_no_cot)} done, {len(todo_no_cot)} remaining")

In [ ]:
sampling_no_cot = SamplingParams(temperature=TEMPERATURE, max_tokens=MAX_ANSWER_TOKENS)

# Rebuild LLM without thinking mode for no_cot
del llm
import gc; gc.collect()
import torch; torch.cuda.empty_cache()

llm = LLM(
    model=MODEL_NAME,
    dtype="bfloat16",
    max_model_len=4096,
)

for i in tqdm(range(0, len(todo_no_cot), CHUNK_SIZE), desc="No-COT baseline"):
    chunk = todo_no_cot[i : i + CHUNK_SIZE]

    prompts = []
    for ex in chunk:
        messages = build_no_cot_messages(ex["question"])
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )
        prompts.append(prompt)

    outputs = llm.generate(prompts, sampling_no_cot)

    for ex, output in zip(chunk, outputs):
        generated_text = output.outputs[0].text
        predicted = extract_predicted_answer(generated_text)

        result = {
            "problem_id": ex["problem_id"],
            "condition": "no_cot",
            "question": ex["question"],
            "gold_answer": ex["gold_answer"],
            "cot_text": "",
            "prefill_text": "",
            "generated_tokens": generated_text,
            "predicted_answer": predicted,
            "error": None,
        }
        (PREFILL_CACHE / f"no_cot_{ex['problem_id']}.json").write_text(json.dumps(result))

print("No-COT baseline complete.")

In [ ]:
# Verify no_cot accuracy
no_cot_results = []
for p in sorted(PREFILL_CACHE.glob("no_cot_*.json"), key=lambda x: int(x.stem.split("_")[-1])):
    no_cot_results.append(json.loads(p.read_text()))

correct_nc = sum(1 for r in no_cot_results if r["predicted_answer"] == r["gold_answer"])
print(f"No-COT accuracy: {correct_nc}/{len(no_cot_results)} = {correct_nc/len(no_cot_results):.1%}")

In [ ]:
# Clean up
del llm
import gc; gc.collect()
import torch; torch.cuda.empty_cache()
print("GPU memory freed. Ready for notebook 03.")